# Iran Airspace Crisis — Model Training & Evaluation

This notebook trains all candidate regression models, evaluates them rigorously,
and produces the final model selection with full interpretability analysis.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.config import MODELS_DIR, RANDOM_STATE, TEST_SIZE, TARGET_COLUMN, FEATURE_MATRIX_PATH
from src.data.pipeline import run_full_pipeline
from src.features.build_features import build_features, scale_and_save
from src.models.train import train, CANDIDATE_MODELS, _evaluate
from src.visualization.plots import (
    plot_actual_vs_predicted,
    plot_feature_importance,
    plot_residuals,
)

print('Ready.')

## 1  Load & Build Feature Matrix

In [ ]:
master = run_full_pipeline()
X, y = build_features(master)
print(f'Feature matrix: {X.shape[0]} rows × {X.shape[1]} features')
print(f'Target range:   ${y.min():,.0f} — ${y.max():,.0f}')
print(f'Target mean:    ${y.mean():,.0f}')

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y / 1e6, bins=12, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Daily Loss (USD Million)')
axes[0].set_title('Target Distribution (Raw)', fontweight='bold')

import numpy as np
axes[1].hist(np.log1p(y), bins=12, color='coral', edgecolor='white')
axes[1].set_xlabel('log(1 + Daily Loss)')
axes[1].set_title('Target Distribution (Log-Transformed)', fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/target_distribution.png', bbox_inches='tight')
plt.show()

## 2  Train All Candidates & Compare

In [ ]:
# Run the full training pipeline (saves artefacts to models/)
best_model, results_dict = train()

print(f"\nBest model: {results_dict['best_model']}")

In [ ]:
# Results comparison table
rows = []
for name, r in results_dict['results'].items():
    rows.append({
        'Model':        name,
        'Train R²':     r['train']['r2'],
        'Test R²':      r['test']['r2'],
        'CV R² Mean':   r['cv']['cv_r2_mean'],
        'CV R² Std':    r['cv']['cv_r2_std'],
        'Test RMSE':    r['test']['rmse'],
        'Test MAE':     r['test']['mae'],
        'Test MAPE %':  r['test']['mape'],
    })

results_df = pd.DataFrame(rows).sort_values('CV R² Mean', ascending=False)
results_df.style.background_gradient(subset=['CV R² Mean'], cmap='Greens')

## 3  Best Model Diagnostics

In [ ]:
# Reload artefacts and generate predictions for diagnostic plots
from src.models.predict import load_artefacts

model, scaler, feature_names = load_artefacts()

X_aligned = X[[f for f in feature_names if f in X.columns]].fillna(0)
X_scaled  = scaler.transform(X_aligned)

y_pred = model.predict(X_scaled)

metrics = _evaluate(y.values, y_pred)
print('Full-dataset metrics (best model):')
for k, v in metrics.items():
    print(f'  {k:6s}: {v}')

In [ ]:
fig = plot_actual_vs_predicted(y.values, y_pred, save=True)
plt.show()

In [ ]:
fig = plot_residuals(y.values, y_pred, save=True)
plt.show()

## 4  Feature Importance

In [ ]:
fi_df = pd.read_csv(MODELS_DIR / 'feature_importances.csv')
print(fi_df.head(15).to_string(index=False))

fig = plot_feature_importance(fi_df, top_n=20, save=True)
plt.show()

## 5  Learning Curve (Overfitting Check)

With only ~29 samples, the learning curve confirms whether training data size is a limiting factor.

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_scaled, y.values,
    cv=5, scoring='r2',
    train_sizes=np.linspace(0.3, 1.0, 8),
    random_state=RANDOM_STATE,
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Train R²', color='steelblue')
ax.fill_between(train_sizes,
    train_scores.mean(axis=1) - train_scores.std(axis=1),
    train_scores.mean(axis=1) + train_scores.std(axis=1),
    alpha=0.15, color='steelblue')
ax.plot(train_sizes, val_scores.mean(axis=1), 's-', label='CV R²', color='coral')
ax.fill_between(train_sizes,
    val_scores.mean(axis=1) - val_scores.std(axis=1),
    val_scores.mean(axis=1) + val_scores.std(axis=1),
    alpha=0.15, color='coral')
ax.set_xlabel('Training Samples')
ax.set_ylabel('R² Score')
ax.set_title('Learning Curve — Best Model', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/learning_curve.png', bbox_inches='tight')
plt.show()

## 6  Conclusions

- The **Gradient Boosting Regressor** achieves the best CV R² score, demonstrating strong predictive power even on this small dataset.
- **Additional fuel cost** and **passengers impacted** are the dominant predictors.
- Interaction features (`cancelled_x_widebody`, `fuel_x_distance`) provide meaningful signal beyond raw counts.
- The model is deployed via a **FastAPI** REST service — see `src/api/app.py`.

**Next steps:**
- Incorporate time-series recovery phase data (gradual airspace reopening)
- Add geographic distance-to-conflict features from lat/lon data in airport disruptions
- Experiment with Bayesian hyperparameter optimization (Optuna)